In [1]:
suppressPackageStartupMessages({
    library(Seurat)
    library(SingleCellExperiment)
    library(data.table)
    library(dplyr)
})

In [2]:
getwd()

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/final"

In [7]:
io = list()
io$gene_metadata = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/Mmusculus_genes_BioMart.87.txt'
io$umap = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/umap.csv'
io$outdir = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/final/'

In [4]:
sce = readRDS('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/embryo_sce.rds')

In [5]:
sce

class: SingleCellExperiment 
dim: 27669 430339 
metadata(0):
assays(1): counts
rownames(27669): ENSMUSG00000051951 ENSMUSG00000089699 ...
  ENSMUSG00000096730 ENSMUSG00000095742
rowData names(0):
colnames(430339): cell_1 cell_2 ... ext_cell_351871 ext_cell_351872
colData names(18): cell sample ...
  celltype_PijuanSala2019_E85mapped_WOTdescendant
  celltype_extended_atlas
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [6]:
# Rename ensemble IDs to gene names in the atlas
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
  .[symbol!="" & ens_id%in%rownames(sce)] %>%
  .[!duplicated(symbol)]

sce <- sce[rownames(sce)%in%gene_metadata$ens_id,]
foo <- gene_metadata$symbol; names(foo) <- gene_metadata$ens_id
rownames(sce) <- foo[rownames(sce)]

In [7]:
sizeFactors(sce) = as.numeric(sizeFactors(sce))
colData(sce)$sizeFactor = sizeFactors(sce)

In [8]:
meta = as.data.table(colData(sce))

In [9]:
keep = lapply(unique(meta$stage), function(x){
  if(x == "mixed_gastrulation"){
    return(c())
  } else if(sum(meta$stage == x) < 15000) {
    return(which(meta$stage == x))
  } else {
    hits = which(meta$stage == x)
    return(sample(hits, 15000))
  }
})
keep = do.call(c, keep)

meta_subset = meta[keep,]

In [10]:
sce_subset = sce[,as.character(meta_subset$cell)]

In [11]:
saveRDS(sce_subset, file.path(io$outdir, 'SingleCellExperiment.rds'))
fwrite(meta_subset, file.path(io$outdir, 'sample_metadata.txt.gz'))

In [21]:
meta_subset = fread(file.path(io$outdir, 'sample_metadata.txt.gz')) %>% 
    .[,idx:=.I]

In [22]:
meta_subset = merge(meta_subset, umap, by='cell') %>%
    .[order(idx)]

In [23]:
head(meta_subset)

cell,sample,embryo_version,stage,stage_mixed_gastrulation_mapped,somite_count,anatomy,S_score,G2M_score,phase,⋯,louvain,leiden,celltype_PijuanSala2019,celltype_PijuanSala2019_E85mapped,celltype_PijuanSala2019_E85mapped_WOTdescendant,celltype_extended_atlas,sizeFactor,idx,umapX,umapY
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,⋯,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>
cell_1,1,Original,E6.5,E6.5,PooledEPooled6Pooled.Pooled5Pooled,Pooled,0.1717105,0.4300413,G2M,⋯,0,6,Epiblast,Epiblast,Epiblast,Epiblast,22117,1,7.441005,16.451050
cell_2,1,Original,E6.5,E6.5,PooledEPooled6Pooled.Pooled5Pooled,Pooled,0.7101008,0.1834096,S,⋯,1,38,Primitive Streak,Primitive Streak,Primitive Streak,Primitive Streak,100549,2,8.233685,16.704405
cell_5,1,Original,E6.5,E6.5,PooledEPooled6Pooled.Pooled5Pooled,Pooled,0.4004477,0.1887521,S,⋯,2,2,ExE ectoderm,ExE ectoderm,ExE ectoderm,ExE ectoderm,64211,3,-4.212557,3.906405
cell_6,1,Original,E6.5,E6.5,PooledEPooled6Pooled.Pooled5Pooled,Pooled,0.4660223,0.2468976,S,⋯,3,3,Epiblast,Epiblast,Epiblast,Epiblast,105010,4,5.691656,17.967155
cell_8,1,Original,E6.5,E6.5,PooledEPooled6Pooled.Pooled5Pooled,Pooled,0.3464951,0.4098396,G2M,⋯,4,6,Epiblast,Epiblast,Epiblast,Epiblast,116315,5,5.491071,16.801476
cell_9,1,Original,E6.5,E6.5,PooledEPooled6Pooled.Pooled5Pooled,Pooled,0.3482208,0.5135100,G2M,⋯,4,6,Epiblast,Epiblast,Epiblast,Epiblast,124077,6,5.836220,15.550345


In [24]:
fwrite(meta_subset, file.path(io$outdir, 'sample_metadata.txt.gz'))